# RTX 5090 — Mamba3 MIMO 40M smoke

Use a RunPod/Vast image with Python 3.11, CUDA 13.0 and PyTorch 2.9.0+cu130. This notebook verifies the public Blackwell MIMO recipe, then the exact Murmur 350M MIMO shape before its 40M training smoke.

In [ ]:
from pathlib import Path
import subprocess, sys
REPO=Path('/workspace/murmur-science')
if not REPO.exists(): subprocess.run(['git','clone','--branch','codex/rtx5090-mimo-smoke','https://github.com/orkrs/murmur-science.git',str(REPO)],check=True)
%cd /workspace/murmur-science
sys.path.insert(0,str(Path.cwd()/'src'))
print('Repository ready:', Path.cwd())

In [ ]:
# Bootstrap PyTorch when the Vast Jupyter image exposes CUDA but omits the Python package.
import importlib.metadata, subprocess, sys
try: installed_torch=importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError: installed_torch=''
if installed_torch != '2.9.0+cu130':
    subprocess.run([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--index-url','https://download.pytorch.org/whl/cu130','torch==2.9.0'],check=True)
    print('PyTorch installed. Restart the Jupyter kernel once, then rerun this notebook from cell 1.')
    raise SystemExit(0)
print('PyTorch wheel is ready:', installed_torch)

In [ ]:
# Hardware gate after the PyTorch bootstrap cell.
import torch
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU is required')
if 'RTX 5090' not in torch.cuda.get_device_name(0): raise RuntimeError('This notebook is reserved for RTX 5090')
if torch.cuda.get_device_capability(0)!=(12,0): raise RuntimeError(f'Expected sm_120, got {torch.cuda.get_device_capability(0)}')
if torch.__version__ != '2.9.0+cu130': raise RuntimeError(f'Use PyTorch 2.9.0+cu130 image, got {torch.__version__}')
if not str(torch.version.cuda).startswith('13.0'): raise RuntimeError(f'Expected CUDA 13.0, got {torch.version.cuda}')
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)

In [ ]:
%pip install -q datasets sentencepiece pyarrow pandas einops ninja
%pip install -q --upgrade 'tilelang==0.1.9' 'apache-tvm-ffi<=0.1.12' 'quack-kernels>=0.3.4' 'triton>=3.5.0' 'nvidia-cutlass-dsl'
import os; os.environ['MAMBA_FORCE_BUILD']='TRUE'
%pip install -q --no-cache-dir --no-deps --force-reinstall --no-build-isolation 'mamba-ssm==2.3.2.post1'
import importlib.metadata
assert importlib.metadata.version('tilelang') == '0.1.9'
assert importlib.metadata.version('mamba-ssm') == '2.3.2.post1'
print('Blackwell MIMO dependencies ready')

In [ ]:
# Gate 1: public Blackwell MIMO recipe — forward + backward
from mamba_ssm.modules.mamba3 import Mamba3
torch.cuda.reset_peak_memory_stats()
reference=Mamba3(d_model=896,d_state=128,headdim=64,is_mimo=True,mimo_rank=4,chunk_size=16,dtype=torch.bfloat16).cuda().train()
x=torch.randn(1,512,896,device='cuda',dtype=torch.bfloat16,requires_grad=True)
loss=reference(x).float().square().mean(); loss.backward(); torch.cuda.synchronize()
assert torch.isfinite(loss)
print({'reference_gate':'passed','peak_vram_gb':round(torch.cuda.max_memory_allocated()/2**30,2)})

In [ ]:
# Gate 2: exact future Murmur-350M recurrent-core shape — rank 2
torch.cuda.reset_peak_memory_stats()
exact=Mamba3(d_model=1792,d_state=128,headdim=64,is_mimo=True,mimo_rank=2,chunk_size=32,dtype=torch.bfloat16).cuda().train()
x=torch.randn(1,1024,1792,device='cuda',dtype=torch.bfloat16,requires_grad=True)
loss=exact(x).float().square().mean(); loss.backward(); torch.cuda.synchronize()
assert torch.isfinite(loss)
print({'murmur_350m_shape_gate':'passed','peak_vram_gb':round(torch.cuda.max_memory_allocated()/2**30,2)})
del reference, exact, x, loss; torch.cuda.empty_cache()

In [ ]:
!python scripts/param_count.py --config configs/rtx5090_mimo_40m_smoke.toml
from murmur.config import load_run_config
config=load_run_config(Path('configs/rtx5090_mimo_40m_smoke.toml'))
assert config.model.mixer=='mamba3_mimo' and config.train.max_tokens==2_000_000 and config.train.bf16
print('Verified 40M MIMO smoke config')

In [ ]:
subprocess.run([sys.executable,'scripts/build_hf_mix.py','--profile','english_smoke','--output','artifacts/english_smoke_corpus','--max-tokens','2400000'],check=True)
!python scripts/train_tokenizer.py --corpus artifacts/english_smoke_corpus/corpus.txt --output artifacts/english_smoke_tokenizer.model --vocab-size 32000
!python scripts/prepare_data.py --config configs/rtx5090_mimo_40m_smoke.toml --tokenizer artifacts/english_smoke_tokenizer.model --train-input artifacts/english_smoke_corpus/train.jsonl --val-input artifacts/english_smoke_corpus/val.jsonl --output artifacts/english_smoke_data

In [ ]:
template=Path('configs/rtx5090_mimo_40m_smoke.toml').read_text()
Path('configs/rtx5090_mimo_40m_smoke_session.toml').write_text(template.replace('artifacts/data/train.bin','artifacts/english_smoke_data/train.bin').replace('artifacts/data/val.bin','artifacts/english_smoke_data/val.bin'))
run_dir=Path('artifacts/runs/rtx5090_mimo_40m_smoke')
subprocess.run([sys.executable,'scripts/train.py','--config','configs/rtx5090_mimo_40m_smoke_session.toml','--run-dir',str(run_dir),'--device','cuda'],check=True)
assert (run_dir/'checkpoints'/'last'/'COMPLETED').exists()
print('RTX 5090 MIMO smoke checkpoint ready')